# 05d — Model D Training: All-Combinations Stacking

Combines race-level predictions from **all** variants of Models A, B, C
into stacking ensembles. Two-phase approach:
- **Phase 1:** RidgeCV screen of all A x B x C combinations
- **Phase 2:** Full tournament on top 20 combinations

Uses out-of-fold (OOF) predictions as meta-features to prevent leakage.

## 0. Setup

In [ ]:
import os
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    # We're likely in notebooks/ — go up to repo root
    for p in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "pyproject.toml").exists():
            os.chdir(p)
            break

print(f"Working directory: {Path.cwd()}")

In [ ]:
import itertools
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb

from f1_predictor.features.splits import LeaveOneSeasonOut
from f1_predictor.data.storage import (
    load_training_parquet,
    save_training_parquet,
    save_model_pickle as gcs_save_model_pickle,
    sync_training_from_gcs,
)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

TRAINING_DIR = Path("data/training")
MODEL_DIR = Path("data/raw/model")
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Sync base model artifacts from GCS
for mt in ["A", "B", "C"]:
    sync_training_from_gcs(mt, TRAINING_DIR)
print("Synced base model artifacts from GCS.")

In [ ]:
def wrap_imputer(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])

NAN_TOLERANT_D = {"XGBoost_shallow", "LightGBM_shallow"}

def cv_evaluate(model, X, y, splitter, groups):
    fold_rmse, fold_mae = [], []
    for train_idx, val_idx in splitter.split(groups):
        import sklearn.base
        m = sklearn.base.clone(model)
        m.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = m.predict(X.iloc[val_idx])
        fold_rmse.append(np.sqrt(mean_squared_error(y.iloc[val_idx], preds)))
        fold_mae.append(mean_absolute_error(y.iloc[val_idx], preds))
    return {
        "fold_rmse": fold_rmse, "fold_mae": fold_mae,
        "mean_rmse": np.mean(fold_rmse), "std_rmse": np.std(fold_rmse),
        "mean_mae": np.mean(fold_mae),
    }

## 1. Discover All Variant Predictions

Scan `data/training/` for all Validation parquets from each model type.

In [ ]:
def discover_variants(model_type):
    """Find all variant names that have Validation parquets."""
    files = sorted(TRAINING_DIR.glob(f"model_{model_type}_*_Validation.parquet"))
    variants = []
    for f in files:
        name = f.stem.replace(f"model_{model_type}_", "").replace("_Validation", "")
        variants.append(name)
    return variants

variants_A = discover_variants("A")
variants_B = discover_variants("B")
variants_C = discover_variants("C")
print(f"Model A variants ({len(variants_A)}): {variants_A}")
print(f"Model B variants ({len(variants_B)}): {variants_B}")
print(f"Model C variants ({len(variants_C)}): {variants_C}")
print(f"Total combinations: {len(variants_A)} × {len(variants_B)} × {len(variants_C)} = "
      f"{len(variants_A) * len(variants_B) * len(variants_C)}")

In [ ]:
def aggregate_lap_to_race(val_df):
    """Aggregate lap-level predictions to race level using the last lap per driver-race."""
    val_df = val_df.sort_values(["season", "round", "driver_abbrev", "lap_number"])
    last_laps = val_df.groupby(["season", "round", "driver_abbrev"]).tail(1)
    return last_laps[["season", "round", "driver_abbrev", "y_true", "y_pred"]].copy()

# Load and cache all variant predictions
merge_key = ["season", "round", "driver_abbrev"]
preds_A, preds_B, preds_C = {}, {}, {}

for v in variants_A:
    df = load_training_parquet(f"model_A_{v}_Validation.parquet", TRAINING_DIR)
    race_df = aggregate_lap_to_race(df)
    preds_A[v] = race_df.rename(columns={"y_pred": f"pred_A_{v}", "y_true": "true_pos"})

for v in variants_B:
    df = load_training_parquet(f"model_B_{v}_Validation.parquet", TRAINING_DIR)
    race_df = aggregate_lap_to_race(df)
    preds_B[v] = race_df.rename(columns={"y_pred": f"pred_B_{v}", "y_true": "true_pos"})

for v in variants_C:
    df = load_training_parquet(f"model_C_{v}_Validation.parquet", TRAINING_DIR)
    preds_C[v] = df.rename(columns={"y_pred": f"pred_C_{v}", "y_true": "true_pos"})

print(f"Loaded {len(preds_A)} A, {len(preds_B)} B, {len(preds_C)} C variant predictions")

## 2. Build All-Variants Meta Matrix

In [ ]:
# Start with first C variant to get the base structure
first_c = variants_C[0]
base = preds_C[first_c][merge_key + ["true_pos", f"pred_C_{first_c}"]].copy()

# Merge all other C variants
for v in variants_C[1:]:
    base = base.merge(preds_C[v][merge_key + [f"pred_C_{v}"]], on=merge_key, how="outer")

# Merge all A variants
for v in variants_A:
    base = base.merge(preds_A[v][merge_key + [f"pred_A_{v}"]], on=merge_key, how="left")

# Merge all B variants
for v in variants_B:
    base = base.merge(preds_B[v][merge_key + [f"pred_B_{v}"]], on=merge_key, how="left")

base = base.dropna(subset=["true_pos"]).reset_index(drop=True)
y_all = base["true_pos"]
groups_all = base["season"].values
print(f"All-variants matrix: {base.shape}")
print(f"Seasons: {sorted(base['season'].unique())}")

## 3. CV Splitter

In [ ]:
available_seasons = sorted(base["season"].unique())
val_seasons = [s for s in available_seasons if s != 2023]
splitter = LeaveOneSeasonOut(val_seasons=val_seasons, test_season=2023)
print(f"Val seasons: {val_seasons}, Test: 2023")
print(f"CV folds: {splitter.get_n_splits()}")

## 4. Phase 1 — RidgeCV Screen of ALL Combinations

Fast screen using RidgeCV on every A x B x C combination.

In [ ]:
all_combos = list(itertools.product(variants_A, variants_B, variants_C))
print(f"Screening {len(all_combos)} combinations with RidgeCV...")

phase1_results = []
for i, (va, vb, vc) in enumerate(all_combos):
    feat_cols = [f"pred_A_{va}", f"pred_B_{vb}", f"pred_C_{vc}"]
    X_combo = base[feat_cols]

    model = wrap_imputer(RidgeCV())
    result = cv_evaluate(model, X_combo, y_all, splitter, groups_all)
    phase1_results.append({
        "A": va, "B": vb, "C": vc,
        "combo": f"{va}__{vb}__{vc}",
        "mean_rmse": result["mean_rmse"],
        "std_rmse": result["std_rmse"],
    })
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(all_combos)} done...")

p1_df = pd.DataFrame(phase1_results).sort_values("mean_rmse").reset_index(drop=True)
print(f"\nPhase 1 complete. Top 10 combinations:")
print(p1_df.head(10)[["combo", "mean_rmse", "std_rmse"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(p1_df["mean_rmse"], bins=40, edgecolor="black", alpha=0.7)
best_val = p1_df["mean_rmse"].iloc[0]
top20_val = p1_df["mean_rmse"].iloc[19]
ax.axvline(best_val, color="red", linestyle="--", label=f"Best: {best_val:.4f}")
ax.axvline(top20_val, color="orange", linestyle="--", label=f"Top-20: {top20_val:.4f}")
ax.set_xlabel("CV RMSE (RidgeCV)")
ax.set_ylabel("Count")
ax.set_title(f"Phase 1: Distribution of {len(all_combos)} Combination RMSEs")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Phase 2 — Full Tournament on Top 20

In [ ]:
TOP_N = 20
top_combos = p1_df.head(TOP_N)
print(f"Phase 2: Full tournament on top {TOP_N} combinations\n")

D_MODEL_CLASSES = {
    "RidgeCV": RidgeCV,
    "LassoCV": LassoCV,
    "ElasticNetCV": ElasticNetCV,
    "XGBoost_shallow": xgb.XGBRegressor,
    "LightGBM_shallow": lgb.LGBMRegressor,
}

def get_d_param_space(name, trial):
    if name == "XGBoost_shallow":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 500),
            max_depth=trial.suggest_int("max_depth", 2, 4),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            random_state=42, verbosity=0,
        )
    elif name == "LightGBM_shallow":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 500),
            max_depth=trial.suggest_int("max_depth", 2, 4),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            random_state=42, verbose=-1,
        )
    elif name == "RidgeCV":
        alphas = [trial.suggest_float(f"alpha_{i}", 0.001, 100.0, log=True) for i in range(5)]
        return dict(alphas=sorted(alphas))
    elif name == "LassoCV":
        return dict(n_alphas=trial.suggest_int("n_alphas", 50, 200), random_state=42, max_iter=5000)
    elif name == "ElasticNetCV":
        return dict(
            l1_ratio=[trial.suggest_float(f"l1_{i}", 0.1, 0.9) for i in range(3)],
            n_alphas=trial.suggest_int("n_alphas", 50, 200), random_state=42, max_iter=5000,
        )
    return {}

def reconstruct_d_params(name, best_params):
    params = dict(best_params)
    if name == "XGBoost_shallow":
        params.update(random_state=42, verbosity=0)
    elif name == "LightGBM_shallow":
        params.update(random_state=42, verbose=-1)
    elif name == "RidgeCV":
        alpha_keys = sorted(k for k in params if k.startswith("alpha_"))
        alphas = sorted(params.pop(k) for k in alpha_keys)
        params["alphas"] = alphas
    elif name == "LassoCV":
        params.update(random_state=42, max_iter=5000)
    elif name == "ElasticNetCV":
        l1_keys = sorted(k for k in params if k.startswith("l1_"))
        l1_ratio = [params.pop(k) for k in l1_keys]
        params["l1_ratio"] = l1_ratio
        params.update(random_state=42, max_iter=5000)
    return params

phase2_results = []
for _, row in top_combos.iterrows():
    va, vb, vc = row["A"], row["B"], row["C"]
    combo_name = row["combo"]
    feat_cols = [f"pred_A_{va}", f"pred_B_{vb}", f"pred_C_{vc}"]
    X_combo = base[feat_cols]

    # Screen 5 meta-learners
    combo_rows = []
    meta_candidates = [
        ("RidgeCV", wrap_imputer(RidgeCV())),
        ("LassoCV", wrap_imputer(LassoCV(random_state=42, max_iter=5000))),
        ("ElasticNetCV", wrap_imputer(
            ElasticNetCV(random_state=42, max_iter=5000))),
        ("XGBoost_shallow", xgb.XGBRegressor(
            n_estimators=100, max_depth=3, random_state=42, verbosity=0)),
        ("LightGBM_shallow", lgb.LGBMRegressor(
            n_estimators=100, max_depth=3, random_state=42, verbose=-1)),
    ]
    for meta_name, meta_model in meta_candidates:
        result = cv_evaluate(meta_model, X_combo, y_all, splitter, groups_all)
        combo_rows.append({"meta": meta_name, "mean_rmse": result["mean_rmse"]})

    best_meta = sorted(combo_rows, key=lambda r: r["mean_rmse"])[0]

    # Optuna tune the best meta-learner (10 trials)
    best_meta_name = best_meta["meta"]
    def objective(trial):
        params = get_d_param_space(best_meta_name, trial)
        model = D_MODEL_CLASSES[best_meta_name](**params)
        if best_meta_name not in NAN_TOLERANT_D:
            model = wrap_imputer(model)
        return cv_evaluate(model, X_combo, y_all, splitter, groups_all)["mean_rmse"]

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=10, show_progress_bar=False)

    phase2_results.append({
        "A": va, "B": vb, "C": vc, "combo": combo_name,
        "meta": best_meta_name, "best_rmse": study.best_value,
        "best_params": study.best_params,
        "phase1_rmse": row["mean_rmse"],
    })
    print(f"  {combo_name}: meta={best_meta_name}, RMSE={study.best_value:.4f}")

p2_df = pd.DataFrame(phase2_results).sort_values("best_rmse").reset_index(drop=True)
print(f"\nPhase 2 complete. Top 5:")
print(p2_df.head()[["combo", "meta", "best_rmse"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(range(TOP_N), p2_df["best_rmse"].values)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels([f"{r['combo']}\n({r['meta']})" for _, r in p2_df.iterrows()], fontsize=7)
ax.set_xlabel("CV RMSE (tuned)")
ax.set_title(f"Phase 2: Top {TOP_N} Combinations After Optuna Tuning")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Test Set Evaluation (Top 5 Combinations)

In [ ]:
train_idx, test_idx = splitter.get_test_split(groups_all)
id_train = base[merge_key].iloc[train_idx]
id_test = base[merge_key].iloc[test_idx]
y_train = y_all.iloc[train_idx]
y_test = y_all.iloc[test_idx]

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")
print(f"Test season: {sorted(base['season'].iloc[test_idx].unique())}")

final_results = []
for _, row in p2_df.head(5).iterrows():
    va, vb, vc = row["A"], row["B"], row["C"]
    feat_cols = [f"pred_A_{va}", f"pred_B_{vb}", f"pred_C_{vc}"]
    X_train_combo = base[feat_cols].iloc[train_idx]
    X_test_combo = base[feat_cols].iloc[test_idx]

    params = reconstruct_d_params(row["meta"], row["best_params"])
    model = D_MODEL_CLASSES[row["meta"]](**params)
    if row["meta"] not in NAN_TOLERANT_D:
        model = wrap_imputer(model)

    model.fit(X_train_combo, y_train)

    test_preds = model.predict(X_test_combo)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    test_mae = mean_absolute_error(y_test, test_preds)

    final_results.append({
        "combo": row["combo"], "meta": row["meta"],
        "val_rmse": row["best_rmse"], "test_rmse": test_rmse, "test_mae": test_mae,
        "overfit_gap": test_rmse - row["best_rmse"],
    })
    print(f"  {row['combo']} ({row['meta']}): val={row['best_rmse']:.4f}, test={test_rmse:.4f}")

final_df = pd.DataFrame(final_results).sort_values("test_rmse").reset_index(drop=True)
final_df

## 7. Save Artifacts (Top 5 Combinations)

In [ ]:
for _, row in p2_df.head(5).iterrows():
    va, vb, vc = row["A"], row["B"], row["C"]
    feat_cols = [f"pred_A_{va}", f"pred_B_{vb}", f"pred_C_{vc}"]
    X_train_combo = base[feat_cols].iloc[train_idx]
    X_test_combo = base[feat_cols].iloc[test_idx]

    params = reconstruct_d_params(row["meta"], row["best_params"])
    model = D_MODEL_CLASSES[row["meta"]](**params)
    if row["meta"] not in NAN_TOLERANT_D:
        model = wrap_imputer(model)
    model.fit(X_train_combo, y_train)

    combo_tag = f"{va}__{vb}__{vc}__{row['meta']}"
    for split_name, X_s, y_s, id_s in [
        ("Training", X_train_combo, y_train, id_train),
        ("Test", X_test_combo, y_test, id_test),
    ]:
        out = id_s.copy()
        out["y_true"] = y_s.values
        out["y_pred"] = model.predict(X_s)
        fname = f"model_D_{combo_tag}_{split_name}.parquet"
        uri = save_training_parquet(out, fname, TRAINING_DIR)
        print(f"  Saved {fname} -> {uri}")

    pkl_name = f"Model_D_{combo_tag}.pkl"
    uri = gcs_save_model_pickle(model, pkl_name, MODEL_DIR)
    print(f"  Saved {pkl_name} -> {uri}")

print("\nDone! All Model D artifacts saved.")

## Summary

In [ ]:
print("=" * 60)
print("MODEL D (ALL-COMBINATIONS STACKING) COMPLETE")
print("=" * 60)
print(f"\nTotal combinations screened: {len(all_combos)}")
print(f"Phase 1 (RidgeCV): {len(all_combos)} combos -> top {TOP_N}")
print(f"Phase 2 (tournament + Optuna): top {TOP_N} -> final 5")
print(f"\nFinal 5 combinations:")
for _, row in final_df.iterrows():
    print(f"  {row['combo']:40s}  meta={row['meta']:15s}  "
          f"test_rmse={row['test_rmse']:.4f}  gap={row['overfit_gap']:.4f}")